# Task 2.2: Implementation with Paper References

**Paper:** Breaking the Curse of Kernelization: Budgeted Stochastic Gradient Descent for Large-Scale SVM Training  
**Authors:** Zhuang Wang, Koby Crammer, Slobodan Vucetic  
**Venue:** JMLR, 2012

---

## Contribution Being Reproduced

I am reproducing the **BPegasos algorithm with all three budget maintenance strategies** (removal, projection, and merging), which together constitute the paper's primary algorithmic contribution. The merging variant (BPegasos+merge) is the best-performing one, and I focus on it while also implementing the other two for comparison.

Specifically, I implement Algorithm 1 (the core BSGD online learning loop) using the Pegasos learning rate from Table 2, and Algorithm 2 (all three budget maintenance strategies as described in Sections 6.1 through 6.3).

**Evaluation metric:** Classification accuracy on a held-out test set, which is the primary metric used throughout the paper (Tables 3 through 5, Section 7).

In [1]:
%matplotlib inline
# ============================================================
# Enhanced Reproducibility Setup & Environment Tracking
# Refers back to the theory steps detailed in task_1_1.ipynb
# ============================================================
import numpy as np
import matplotlib as mpl
import sklearn
import sys
import os

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

print("=== Environment Set up for Reproducibility ===")
print(f"Python Version: {sys.version.split()[0]}")
print(f"NumPy Version: {np.__version__}")
print(f"Matplotlib Version: {mpl.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")
print(f"Global Random Seed: {RANDOM_SEED}")
print("==============================================")
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
import time
import matplotlib.pyplot as plt


=== Environment Set up for Reproducibility ===
Python Version: 3.13.1
NumPy Version: 2.2.3
Matplotlib Version: 3.10.7
Scikit-Learn Version: 1.7.2
Global Random Seed: 42


In [2]:
# ============================================================
# Dataset generation (same as Task 2.1)
# ============================================================
def generate_checkerboard(n_samples=2000, grid_size=4, noise=0.0, random_state=42):
    rng = np.random.RandomState(random_state)
    X = rng.uniform(0, 1, size=(n_samples, 2))
    cell_x = np.floor(X[:, 0] * grid_size).astype(int)
    cell_y = np.floor(X[:, 1] * grid_size).astype(int)
    y = np.where((cell_x + cell_y) % 2 == 0, 1, -1)
    if noise > 0:
        X += rng.normal(0, noise, size=X.shape)
    return X, y

X, y = generate_checkerboard(n_samples=2000, grid_size=4, random_state=RANDOM_SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y
)
print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")

Train: 1400 samples, Test: 600 samples


---

## Part 1: RBF Kernel

The Gaussian RBF kernel is defined as $k(x, x') = \exp(-\|x - x'\|^2 / 2\sigma^2)$ where $\sigma$ is the kernel width. The paper uses this kernel throughout all experiments (Section 7.1). One important property is that $k(x,x) = 1$ for all $x$, which means $\|\Phi(x)\| = 1$. This satisfies the bounded norm assumption required by Theorems 1 through 3.

In [3]:
def rbf_kernel(x1, x2, sigma):
    """
    RBF (Gaussian) kernel between two vectors.
    k(x1, x2) = exp(-||x1 - x2||^2 / (2 * sigma^2))
    
    Reference: Section 7.1 of Wang et al. (2012)
    """
    diff = x1 - x2
    return np.exp(-np.dot(diff, diff) / (2 * sigma ** 2))

# Quick verification that k(x,x) = 1
print(f"k(x, x) = {rbf_kernel(X_train[0], X_train[0], sigma=0.1):.4f}  (should be 1.0)")
print(f"k(x, x') = {rbf_kernel(X_train[0], X_train[1], sigma=0.1):.6f}  (should be < 1.0)")

k(x, x) = 1.0000  (should be 1.0)
k(x, x') = 0.000000  (should be < 1.0)


The RBF kernel maps each point to an infinite-dimensional feature space where $k(x, x) = 1$ always. This ensures the bounded norm condition required by the convergence analysis (Theorems 1 through 3). The kernel width $\sigma$ controls how localised the kernel is: smaller $\sigma$ makes the kernel sharper and more sensitive to small distances.

---

## Part 2: BPegasos Implementation

The following class implements the full BPegasos algorithm as described in Algorithms 1 and 2 of the paper. The SGD update follows the Pegasos learning rate $\eta_t = 1/(\lambda t)$ from Table 2. At each time step, all existing coefficients are decayed by $(1 - \eta_t \lambda)$ to account for regularisation (Algorithm 1, Lines 4 through 6), and if the hinge loss is positive on the current example, it is added as a new support vector (Lines 7 through 11). When the SV count exceeds the budget, one of three maintenance strategies is applied (Line 13).

In [4]:
class BPegasos:
    """
    Budgeted Pegasos (BPegasos) from Wang et al. (2012).
    
    Implements Algorithm 1 (online kernel SVM with budget) and
    Algorithm 2 (three budget maintenance strategies).
    
    Parameters
    ----------
    budget : int
        Maximum number of support vectors (B in the paper).
    lam : float
        Regularisation parameter (lambda in Eq. 1).
    sigma : float
        Gaussian kernel width.
    strategy : str
        Budget maintenance: 'remove', 'project', or 'merge'.
    """
    
    def __init__(self, budget=100, lam=1e-4, sigma=0.125, strategy='merge'):
        self.budget = budget
        self.lam = lam
        self.sigma = sigma
        self.strategy = strategy
        self.sv_X = []       # Support vector positions
        self.sv_alpha = []   # Coefficients (alpha_j)
        self.t = 0           # Global time step
    
    def _kernel(self, x1, x2):
        """RBF kernel."""
        diff = x1 - x2
        return np.exp(-np.dot(diff, diff) / (2 * self.sigma ** 2))
    
    def _predict_value(self, x):
        """
        Decision function: f(x) = sum_j alpha_j * k(x_j, x)
        This is the kernel representation of w^T Phi(x) from Section 3.
        """
        value = 0.0
        for j in range(len(self.sv_X)):
            value += self.sv_alpha[j] * self._kernel(self.sv_X[j], x)
        return value
    
    def _budget_maintenance_remove(self):
        """
        Removal strategy (Section 6.1, Algorithm 2).
        
        Removes the SV with smallest |alpha|. For RBF kernel where
        k(x,x) = 1, this is the SV contributing least to the decision.
        """
        min_idx = np.argmin([abs(a) for a in self.sv_alpha])
        del self.sv_X[min_idx]
        del self.sv_alpha[min_idx]
    
    def _budget_maintenance_project(self):
        """
        Projection strategy (Section 6.2, Equation 14, Algorithm 2).
        
        Before removing the smallest-weight SV, project its contribution
        onto the remaining SVs using the kernel matrix inverse. This
        distributes the removed SV's weight to the survivors.
        """
        n_sv = len(self.sv_X)
        p = np.argmin([abs(a) for a in self.sv_alpha])
        remaining = [i for i in range(n_sv) if i != p]
        if len(remaining) == 0:
            del self.sv_X[p]
            del self.sv_alpha[p]
            return
        
        # Build kernel matrix K_p of remaining SVs
        K_p = np.zeros((len(remaining), len(remaining)))
        for i, ri in enumerate(remaining):
            for j, rj in enumerate(remaining):
                K_p[i, j] = self._kernel(self.sv_X[ri], self.sv_X[rj])
        K_p += 1e-8 * np.eye(len(remaining))  # Numerical stability
        
        # Kernel vector between removed SV and remaining SVs
        k_p = np.array([self._kernel(self.sv_X[p], self.sv_X[ri]) for ri in remaining])
        
        # Eq. 14: delta_alpha = alpha_p * K_p^{-1} * k_p
        try:
            proj_coeffs = self.sv_alpha[p] * np.linalg.solve(K_p, k_p)
        except np.linalg.LinAlgError:
            del self.sv_X[p]
            del self.sv_alpha[p]
            return
        
        for i, ri in enumerate(remaining):
            self.sv_alpha[ri] += proj_coeffs[i]
        del self.sv_X[p]
        del self.sv_alpha[p]
    
    def _budget_maintenance_merge(self):
        """
        Merging strategy (Section 6.3, Equations 15-17, Algorithm 2).
        
        Finds the two closest SVs (highest kernel value), replaces them
        with a single SV at a weighted midpoint.
        z = h * x_m + (1 - h) * x_n, where h = |alpha_m| / (|alpha_m| + |alpha_n|)
        alpha_z = alpha_m + alpha_n
        """
        n_sv = len(self.sv_X)
        if n_sv < 2:
            return
        
        # Find closest pair (highest kernel value)
        best_k = -1.0
        best_m, best_n = 0, 1
        for i in range(n_sv):
            for j in range(i + 1, n_sv):
                k_val = self._kernel(self.sv_X[i], self.sv_X[j])
                if k_val > best_k:
                    best_k = k_val
                    best_m, best_n = i, j
        
        m, n = best_m, best_n
        abs_alpha_m = abs(self.sv_alpha[m])
        abs_alpha_n = abs(self.sv_alpha[n])
        denom = abs_alpha_m + abs_alpha_n
        h = abs_alpha_m / denom if denom > 1e-12 else 0.5
        
        z = h * self.sv_X[m] + (1 - h) * self.sv_X[n]
        alpha_z = self.sv_alpha[m] + self.sv_alpha[n]
        
        for idx in sorted([m, n], reverse=True):
            del self.sv_X[idx]
            del self.sv_alpha[idx]
        self.sv_X.append(z)
        self.sv_alpha.append(alpha_z)
    
    def _budget_maintenance(self):
        """Apply the selected budget maintenance strategy."""
        if self.strategy == 'remove':
            self._budget_maintenance_remove()
        elif self.strategy == 'project':
            self._budget_maintenance_project()
        elif self.strategy == 'merge':
            self._budget_maintenance_merge()
        else:
            raise ValueError(f"Unknown strategy: {self.strategy}")
    
    def fit(self, X, y, n_epochs=1, verbose=True):
        """
        Train BPegasos on a data stream (Algorithm 1).
        
        For each example (x_t, y_t):
          1. Decay all SV coefficients: alpha_j *= (1 - eta_t * lambda)
          2. Check hinge loss; if positive, add x_t as new SV
          3. If |SVs| > budget, apply budget maintenance
        """
        n_samples = X.shape[0]
        for epoch in range(n_epochs):
            indices = np.random.permutation(n_samples)
            for i in indices:
                self.t += 1
                x_t, y_t = X[i], y[i]
                
                # Pegasos learning rate (Table 2)
                eta_t = 1.0 / (self.lam * self.t)
                
                # Coefficient decay (Algorithm 1, Lines 4-6)
                decay = 1.0 - eta_t * self.lam
                for j in range(len(self.sv_alpha)):
                    self.sv_alpha[j] *= decay
                
                # Hinge loss check (Algorithm 1, Lines 7-11)
                f_t = self._predict_value(x_t)
                if y_t * f_t < 1.0:
                    self.sv_X.append(x_t.copy())
                    self.sv_alpha.append(eta_t * y_t)
                
                # Budget maintenance (Algorithm 1, Line 13)
                while len(self.sv_X) > self.budget:
                    self._budget_maintenance()
            
            if verbose:
                print(f"  Epoch {epoch+1}/{n_epochs} complete: {len(self.sv_X)} SVs in model")
    
    def predict(self, X):
        """Predict labels: sign(f(x))."""
        return np.array([1 if self._predict_value(X[i]) >= 0 else -1 for i in range(X.shape[0])])
    
    def score(self, X, y):
        """Classification accuracy."""
        return np.mean(self.predict(X) == y)

The BPegasos class above implements the complete algorithm from the paper. Each method maps directly to a component described in the paper:

`_predict_value` computes the kernel expansion $f(x) = \sum_j \alpha_j k(x_j, x)$ from Section 3. `_budget_maintenance_remove` implements Section 6.1, `_budget_maintenance_project` implements Section 6.2 with Equation 14, and `_budget_maintenance_merge` implements Section 6.3 with Equation 17. The `fit` method follows Algorithm 1's outer loop, applying the Pegasos-specific learning rate from Table 2.

---

## Part 3: Training with All Three Strategies

I now train BPegasos with each of the three budget maintenance strategies using the same hyperparameters and random seed, so the only difference between runs is the budget maintenance approach. The hyperparameters are based on the paper's experimental setup (Section 7.1): $\lambda = 10^{-4}$ for regularisation, $\sigma = 0.125$ for the kernel width (which gives $\gamma = 32$ in scikit-learn's convention), and $B = 50$ for the budget.

In [5]:
# ============================================================
# Hyperparameters (all defined in one place)
# ============================================================
BUDGET = 50          # Maximum support vectors (B)
LAMBDA = 1e-4        # Regularisation (lambda in Eq. 1)
SIGMA = 0.125        # RBF kernel width (Section 7.1)
N_EPOCHS = 5         # Multi-epoch training (Section 7.8)

print("Hyperparameters:")
print(f"  Budget (B)      = {BUDGET}")
print(f"  Lambda          = {LAMBDA}")
print(f"  Kernel width    = {SIGMA}")
print(f"  Epochs          = {N_EPOCHS}")
print(f"  Training size   = {X_train.shape[0]}")
print(f"  Test size       = {X_test.shape[0]}")

Hyperparameters:
  Budget (B)      = 50
  Lambda          = 0.0001
  Kernel width    = 0.125
  Epochs          = 5
  Training size   = 1400
  Test size       = 600


In [6]:
# ============================================================
# Train BPegasos with all three strategies
# ============================================================
strategies = ['remove', 'project', 'merge']
results = {}

for strategy in strategies:
    print(f"\nTraining BPegasos+{strategy} (B={BUDGET})...")
    
    np.random.seed(RANDOM_SEED)
    model = BPegasos(budget=BUDGET, lam=LAMBDA, sigma=SIGMA, strategy=strategy)
    
    start_time = time.time()
    model.fit(X_train, y_train, n_epochs=N_EPOCHS, verbose=True)
    train_time = time.time() - start_time
    
    results[strategy] = {
        'train_acc': model.score(X_train, y_train),
        'test_acc': model.score(X_test, y_test),
        'n_svs': len(model.sv_X),
        'train_time': train_time,
        'model': model
    }
    
    r = results[strategy]
    print(f"  Train acc: {r['train_acc']:.4f}, Test acc: {r['test_acc']:.4f}, SVs: {r['n_svs']}, Time: {r['train_time']:.1f}s")


Training BPegasos+remove (B=50)...
  Epoch 1/5 complete: 50 SVs in model


  Epoch 2/5 complete: 50 SVs in model


  Epoch 3/5 complete: 50 SVs in model
  Epoch 4/5 complete: 50 SVs in model


  Epoch 5/5 complete: 50 SVs in model


  Train acc: 0.6550, Test acc: 0.6550, SVs: 50, Time: 0.5s

Training BPegasos+project (B=50)...


...

  Epoch 1/5 complete: 50 SVs in model


  Epoch 2/5 complete: 50 SVs in model


  Epoch 3/5 complete: 50 SVs in model


  Epoch 4/5 complete: 50 SVs in model


  Epoch 5/5 complete: 50 SVs in model
  Train acc: 0.8179, Test acc: 0.8217, SVs: 50, Time: 3.3s


Each strategy is trained with the same random seed so the shuffling order and initial conditions are identical. The only variable that differs is the budget maintenance algorithm that fires when the SV count exceeds $B = 50$. This mirrors the comparative evaluation approach used in the paper's Tables 3 through 5 and Figure 5.

---

## Part 4: Baselines

For context I also train two baselines: unbounded Pegasos (the paper's primary reference) and scikit-learn's SVC (equivalent to LIBSVM, which also appears in the paper's experiments).

In [7]:
from sklearn.svm import SVC

# Unbounded Pegasos (no budget constraint)
print("Training unbounded Pegasos (no budget)...")
np.random.seed(RANDOM_SEED)
pegasos = BPegasos(budget=X_train.shape[0]+1, lam=LAMBDA, sigma=SIGMA, strategy='remove')
start = time.time()
pegasos.fit(X_train, y_train, n_epochs=1, verbose=True)
peg_time = time.time() - start
peg_acc = pegasos.score(X_test, y_test)
print(f"  Test acc: {peg_acc:.4f}, SVs: {len(pegasos.sv_X)}, Time: {peg_time:.1f}s")

# LIBSVM via scikit-learn SVC
print("\nTraining LIBSVM (scikit-learn SVC)...")
gamma = 1.0 / (2 * SIGMA ** 2)
svc = SVC(kernel='rbf', gamma=gamma, C=1.0/(LAMBDA * X_train.shape[0]), random_state=RANDOM_SEED)
start = time.time()
svc.fit(X_train, y_train)
svc_time = time.time() - start
svc_acc = svc.score(X_test, y_test)
print(f"  Test acc: {svc_acc:.4f}, SVs: {svc.n_support_.sum()}, Time: {svc_time:.1f}s")

Training unbounded Pegasos (no budget)...


  Epoch 1/1 complete: 256 SVs in model
  Test acc: 0.9033, SVs: 256, Time: 0.3s

Training LIBSVM (scikit-learn SVC)...


  Test acc: 0.9683, SVs: 297, Time: 0.0s


The unbounded Pegasos baseline uses the same SGD code but with a budget so large that it never triggers. The LIBSVM comparison uses scikit-learn's SVC, converting the paper's kernel width $\sigma$ to scikit-learn's $\gamma = 1/(2\sigma^2)$ and setting $C = 1/(\lambda N)$ to match the paper's regularisation.

---

## Part 5: Summary

In [8]:
print("\n" + "=" * 70)
print(f"{'Method':<25} {'Train Acc':>10} {'Test Acc':>10} {'# SVs':>8} {'Time (s)':>10}")
print("=" * 70)
for s in strategies:
    r = results[s]
    print(f"{'BPegasos+' + s:<25} {r['train_acc']:>10.4f} {r['test_acc']:>10.4f} {r['n_svs']:>8} {r['train_time']:>10.2f}")
print(f"{'Pegasos (unbounded)':<25} {pegasos.score(X_train, y_train):>10.4f} {peg_acc:>10.4f} {len(pegasos.sv_X):>8} {peg_time:>10.2f}")
print(f"{'LIBSVM (sklearn SVC)':<25} {svc.score(X_train, y_train):>10.4f} {svc_acc:>10.4f} {svc.n_support_.sum():>8} {svc_time:>10.2f}")
print("=" * 70)


Method                     Train Acc   Test Acc    # SVs   Time (s)
BPegasos+remove               0.6550     0.6550       50       0.48
BPegasos+project              0.9264     0.9167       50       3.89
BPegasos+merge                0.8179     0.8217       50       3.32


Pegasos (unbounded)           0.8979     0.9033      256       0.31
LIBSVM (sklearn SVC)          0.9793     0.9683      297       0.02


This summary table is analogous to Tables 3 through 5 in the paper. It allows direct comparison of accuracy, model size (number of SVs), and training time across all BPegasos variants and baselines.